# Autocallable

Phoenix autocallable on TICKER1, priced with `AutocallableMCPricer` on the local vol model built from the market.
The note is split in two legs:

$$\text{Price upfront} = \underbrace{\text{PV funding leg}}_{\text{callable floating bond}} - \underbrace{\text{PV option leg}}_{\text{autocallable phoenix}}$$

Docs: *Models and Pricing* (Products: Autocallable; Models: Local volatility).

In [ ]:
import os
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorquant as tq

## Market
Latest snapshot in `marketdata/`: TICKER1 (spot, cash dividends, vol surface), ESTR discount curve, EURIBOR 6M curve.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
estr = mkt.get_ir_curve(tq.Currency.EUR)             # discount curve
eur6m = mkt.get_ir_curve(tq.Currency.EUR, "6M")      # EURIBOR 6M projection curve
print(f"{EVALUATION_DATE}  {TICKER} spot {float(eq.spot):.2f}")

## Product
Strike $K_0$ = initial fixing = spot; barriers in % of $K_0$. On each coupon date $t_i$ (still alive):
coupon $c_i(1 + m_i)\,\mathbb 1\{S_{t_i} \ge B_c K_0\}$ ($m_i$ unpaid coupons, 0 without memory). On each autocall date:
early redemption at par if $S \ge B_a K_0$. At maturity (not called) the investor bears the put
$-p\,(K_0 - S_T)^+/K_0$ if $S_T < B_{\text{put}} K_0$.

| Parameter | Value |
|---|---|
| Maturity | 5Y, semi-annual coupons |
| Coupon | 4% per period, barrier 80%, no memory |
| Autocall | from 1Y, semi-annual, barrier 100% |
| Put | barrier 80%, participation 100% |

In [ ]:
NOTIONAL = 100e6
calendar = tq.TARGET()
mf = tq.BusinessDayConvention.ModifiedFollowing
schedule = tq.ScheduleGenerator(calendar, mf)
end_date = calendar.advance(EVALUATION_DATE, 5, tq.TimeUnit.Years, mf)
coupon_dates = schedule.generate(EVALUATION_DATE, end_date, 6, tq.TimeUnit.Months)[1:]
autocall_dates = schedule.generate(calendar.advance(EVALUATION_DATE, 1, tq.TimeUnit.Years, mf), end_date,
                                   6, tq.TimeUnit.Months)

product = tq.AutocallableOption(
    ccy=tq.Currency.EUR, notional=NOTIONAL, start_date=EVALUATION_DATE, end_date=end_date,
    strike=round(float(eq.spot), 2),
    coupon_fixing_dates=coupon_dates, coupon_payment_dates=coupon_dates,
    coupon_rates=[4.0] * len(coupon_dates), coupon_barriers=[80.0] * len(coupon_dates), memory=False,
    autocall_fixing_dates=autocall_dates, autocall_payment_dates=autocall_dates,
    autocall_barrier=[100.0] * len(autocall_dates),
    payoff_barrier=80.0, payoff_participation=1.0, payoff_type="put",
    underlying=TICKER,
)

## Option leg (local vol)
`AutocallableMCPricer` builds `LocalVolatilityModel.from_equity` from the market and simulates on the fixings merged with
a weekly grid. Investor's side: coupons $-$ put. `barrier_smoothing=0` prices the exact (hard) barriers.

In [ ]:
pricer = tq.AutocallableMCPricer(model=tq.LocalVolatilityModel, n_paths=50_000, seed=42, barrier_smoothing=0.0)
pricer.price(product, mkt)
option_pv = float(product.price) / NOTIONAL
print(f"Option leg PV (% notional): {option_pv:.2%}  +/- {float(product.standard_error) / NOTIONAL:.2%}")
print(f"  coupons : {float(product.coupon_pv):.2%}")
print(f"  put     : {-float(product.put_pv):.2%}")

## Autocall probabilities
Same paths (`pricer.paths` at `pricer.date_grid`): probability of being called on each autocall date.

In [ ]:
paths = pricer.paths.numpy()
col = {d: j for j, d in enumerate(pricer.date_grid)}
alive = np.ones(paths.shape[0], dtype=bool)
rows = []
for d, b in zip(product.autocall_fixing_dates, product.autocall_barrier):
    called = alive & (paths[:, col[d]] >= b / 100 * product.strike)
    rows.append((d, f"{called.mean():.2%}"))
    alive = alive & ~called
print(pd.DataFrame(rows, columns=["date", "P(called)"]).to_string(index=False))
print(f"not called: {alive.mean():.2%}")

## Funding leg (callable floating bond)
EURIBOR 6M forwards (6M curve) on the coupon periods, Act/360 accrual, discounted on ESTR, paid while the note is alive:
a coupon on an autocall date is paid, the following ones are not.

In [ ]:
act360 = tq.DayCounter(tq.DayCounterConvention.Actual360)
alive = np.ones(paths.shape[0], dtype=bool)
funding_pv, start, flows = 0.0, EVALUATION_DATE, []
for d in product.coupon_payment_dates:
    fwd = float(eur6m.forward_rate(start, d))
    tau = act360.year_fraction(start, d)
    df = float(estr.discount(d))
    funding_pv += alive.mean() * fwd * tau * df
    flows.append((start, d, fwd, tau, df, alive.mean()))
    if d in product.autocall_fixing_dates:    # autocall check after the coupon
        i = product.autocall_fixing_dates.index(d)
        alive = alive & (paths[:, col[d]] < product.autocall_barrier[i] / 100 * product.strike)
    start = d
print(pd.DataFrame(flows, columns=["start", "pay", "forward", "tau", "P(0,pay)", "P(alive)"]).round(4).to_string(index=False))
print(f"\nFunding leg PV (% notional): {funding_pv:.2%}")

## Price upfront

In [ ]:
print(f"Funding leg PV : {funding_pv:7.2%}")
print(f"Option leg PV  : {option_pv:7.2%}")
print(f"Price upfront  : {funding_pv - option_pv:7.2%}")